# Chapter-9：多智能体失败归因与评估（Who&When 格式 + AutoFA）

> Last updated: 2026-11-23 ｜ 作者：zhanghong

## 学习目标

第 8 章实现了以 LangGraph 固定图为中心编排、由 A2A 领域智能体与 MCP 工具共同支撑的旅行多智能体系统，并记录了完整的 OpenTelemetry span 轨迹。当系统给出错误结论时，最终答复本身不足以说明责任归属：偏差可能源于任务规划、依赖分析、某个领域智能体的工具调用，也可能源于最终汇总。本章引入 AutoFA（Automated Failure Attribution）方法：先将运行轨迹转换为 Who&When 格式，再由大模型判定哪个智能体（Who）在哪一步（When）首次引入实质性错误，并说明原因（Why）。

本章以两条已记录的旅行轨迹为样例，重点理解三件事：

1. 轨迹转换：将 OpenTelemetry span JSONL 转换为 Who&When JSON，并明确 history 的步骤顺序与编号规则。
2. 失败归因：实现 `all_at_once`、`step_by_step` 与 `binary_search` 三种归因策略，并复用仓库根目录的模型配置。
3. 结果评估：在具备人工标注时，将模型预测与标注逐条比较，计算 Agent 与 Step 的命中率。

详细命令行说明见 [`testdata/README.md`](testdata/README.md)。


## 0. 环境准备与前置条件

使用书仓库统一的 `.venv`，并从仓库根目录 `.env` 加载模型配置。

```bash
pip install openai python-dotenv tqdm
```

本章至少需要在仓库根目录 `.env` 中配置：

```env
DASHSCOPE_API_KEY=sk-你的DashScope密钥
# 可选：复现 2026-06-15 的样例轨迹时设为 2026-06-15
# FA_REFERENCE_DATE=2026-06-15
```

Key 获取：[DashScope 控制台](https://dashscope.console.aliyun.com/)

> `Chapter-9/Automated_FA/.env.example` 是上游 Who&When 代码的原样文件，本章不再单独维护；模型配置以仓库根目录 `.env` 为准。

**轨迹文件要求（JSONL，每行一个 span）：**

转换脚本按后缀匹配 span 名称，不依赖固定前缀：

- `*.planner.*`（排除 `.build_plan` 包装 span）
- `*.agent.invoke`
- `*.orchestration.aggregate`

> **Step 2 归因**需要 API Key；**Step 1 转换**与**Step 4 评估**（复用已有 `outputs/*.txt`）不需要 Key。


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass


def _resolve_chapter_root() -> Path:
    """Resolve the Chapter-9 directory when the notebook runs from either level."""
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "Automated_FA").exists() and (base / "testdata" / "convert_spans_to_whowhen.py").exists():
            return base
        chapter9 = base / "Chapter-9"
        if (chapter9 / "Automated_FA").exists() and (chapter9 / "testdata" / "convert_spans_to_whowhen.py").exists():
            return chapter9
    return Path.cwd()


CHAPTER_ROOT = _resolve_chapter_root()
REPO_ROOT = CHAPTER_ROOT.parent

FA_DIR = CHAPTER_ROOT / "Automated_FA"
TESTDATA_DIR = CHAPTER_ROOT / "testdata"
CONVERTED_DIR = TESTDATA_DIR / "converted"
OUTPUT_DIR = FA_DIR / "outputs"

for p in (REPO_ROOT, FA_DIR, TESTDATA_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Reuse the repository-level .env and llm module for model configuration.
from llm import load_project_env

try:
    ENV_PATH = load_project_env()
    print("仓库根目录 .env:", ENV_PATH)
except FileNotFoundError as exc:
    ENV_PATH = None
    print(exc)

print("Chapter-9 根目录:", CHAPTER_ROOT.resolve())
print("DASHSCOPE_API_KEY:", "已配置" if os.getenv("DASHSCOPE_API_KEY", "").strip() else "未配置")


## 整体流程

```
spans_*.jsonl          convert_spans_to_whowhen.py          converted/*.json
(系统运行轨迹)    ──►   (Step 1 格式转换)              ──►   (归因输入)
                                                              │
                                                              ▼
                                                    Automated_FA/inference.py
                                                              │
                                                              ▼
                                                    outputs/*.txt
                                                    (Agent / Step / Reason)
                                                              │
                                                              ▼ (可选)
                                                    Automated_FA/evaluate.py
                                                    (与人工标注对比准确率)
```

**history 步骤顺序（转换规则）：**

1. **Planner**：所有 `.planner.*` span，按 `start_time` 排序 → `Planner.{步骤名}`
2. **Agent**：所有 `.agent.invoke` span，按 `execution.order` 排序 → `attributes.task.agent`
3. **汇总**：`.orchestration.aggregate` → `Orchestrator`

**Step 编号**等于 `history` 数组下标，从 **0** 开始。转换后的 JSON 使用 `history[i].name` 标识 agent，因此归因时 `--is_handcrafted` 必须设为 **False**。

## 1. 核心流程函数：三阶段归因与可选评估

该函数将 `convert_spans_to_whowhen`、`inference.py` 与 `evaluate.py` 封装为统一流程，与 README 第五节「手动验证流程」逐项对应。

In [ ]:
import re

from convert_spans_to_whowhen import convert_jsonl_to_whowwhen
from evaluate import evaluate_accuracy

METHOD_CHOICES = ("all_at_once", "step_by_step", "binary_search")
DEFAULT_METHOD = "step_by_step"
DEFAULT_MODEL = "qwen-plus"
IS_HANDCRAFTED = False  # Converted JSON uses history[i].name, so this must stay False.


def parse_attribution_results(eval_text: str) -> list[dict[str, Any]]:
    """Parse one outputs/*.txt attribution report into per-file records."""
    results: list[dict[str, Any]] = []
    pattern = r"Prediction for ([^\n:]+\.json)\s*[：:](.*?)(?=Prediction for|\Z)"
    for match in re.finditer(pattern, eval_text, re.DOTALL):
        fname = match.group(1).strip()
        block = match.group(2)
        agent_m = re.search(r"(?:Agent Name|错误智能体)[：:]\s*([\w.]+)", block, re.I)
        step_m = re.search(r"(?:Step Number|步骤编号|步骤)[：:]\s*(\d+)", block, re.I)
        reason_m = re.search(
            r"(?:错误原因|Reason provided by LLM|Reason for Mistake)[：:]\s*(.+)",
            block,
            re.I | re.DOTALL,
        )
        results.append({
            "file": fname,
            "agent": agent_m.group(1) if agent_m else "未知",
            "step": step_m.group(1) if step_m else "未知",
            "reason": reason_m.group(1).strip() if reason_m else "（未解析到原因）",
        })
    return results


def print_attribution_summary_zh(
    eval_file: Path,
    converted_dir: Path | None = None,
) -> list[dict[str, Any]]:
    """Print a Chinese summary of the attribution report."""
    if not eval_file.exists():
        print(f"尚未生成归因输出：{eval_file}")
        print("请先运行 Step 2，或执行 inference.py。")
        return []

    eval_text = eval_file.read_text(encoding="utf-8")
    results = parse_attribution_results(eval_text)
    if not results:
        print("未在输出文件中找到归因结果（需包含「Prediction for xxx.json」段落）。")
        return []

    print(f"归因结果文件：{eval_file}\n")
    for i, item in enumerate(results, start=1):
        question = ""
        if converted_dir:
            json_path = converted_dir / item["file"]
            if json_path.exists():
                question = json.loads(json_path.read_text(encoding="utf-8")).get("question", "")

        print(f"{'─' * 60}")
        print(f"【轨迹 {i}】{item['file']}")
        if question:
            q_preview = question.replace("\n", " ")[:80]
            print(f"  用户问题：{q_preview}{'…' if len(question) > 80 else ''}")
        print(f"  错误智能体：{item['agent']}")
        print(f"  错误步骤：{item['step']}（history 下标，从 0 开始）")
        print(f"  错误原因：{item['reason']}")
    print(f"{'─' * 60}")
    print(f"共 {len(results)} 条轨迹完成归因。")
    return results


def _print_conversion_summary(out_path: Path) -> None:
    with out_path.open("r", encoding="utf-8") as f:
        record = json.load(f)
    print(f"Wrote {out_path}")
    print(f"execution_order: {record.get('execution_order', [])}")
    print(f"history steps: {len(record['history'])}")
    for idx, step in enumerate(record["history"]):
        print(f"  Step {idx}: {step['name']}")


def _run_inference(method: str, model: str, directory_path: Path) -> Path:
    """Run inference.py in a subprocess so the output matches the CLI."""
    output_path = OUTPUT_DIR / f"{method}_{model}_alg_generated.txt"
    cmd = [
        sys.executable,
        str(FA_DIR / "inference.py"),
        "--method", method,
        "--model", model,
        "--is_handcrafted", "False",
        "--directory_path", str(directory_path),
    ]
    print("运行:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(FA_DIR), check=True)
    if not output_path.exists():
        raise FileNotFoundError(f"归因未生成输出: {output_path}")
    return output_path


def print_evaluation_comparison_zh(
    predictions: dict[str, Any],
    converted_dir: Path,
    total_files: int,
) -> None:
    """Compare predictions with labels using the same rule as evaluate_accuracy."""
    print("\n【预测 vs 标注 对比】")
    labeled_count = 0
    correct_agent = 0
    correct_step = 0

    for fname, pred in predictions.items():
        json_path = converted_dir / fname
        actual_agent, actual_step = None, None
        if json_path.exists():
            data = json.loads(json_path.read_text(encoding="utf-8"))
            actual_agent = data.get("mistake_agent")
            actual_step = data.get("mistake_step")

        pred_agent = pred["predicted_agent"]
        pred_step = pred["predicted_step"]

        if actual_agent is None or actual_step is None:
            print(f"  {fname}")
            print(f"    预测：{pred_agent} @ 步骤 {pred_step}")
            print(f"    标注：无（未写入 mistake_agent / mistake_step，不参与计分）")
            continue

        labeled_count += 1
        agent_ok = str(actual_agent) in str(pred_agent)
        step_ok = str(actual_step) in str(pred_step)
        if agent_ok:
            correct_agent += 1
        if step_ok:
            correct_step += 1

        mark = "✓" if (agent_ok and step_ok) else "✗"
        print(f"  {mark} {fname}")
        print(f"    预测：{pred_agent} @ 步骤 {pred_step}")
        print(f"    标注：{actual_agent} @ 步骤 {actual_step}")
        if not agent_ok:
            print(f"    → Agent 不匹配")
        if not step_ok:
            print(f"    → 步骤不匹配")

    print(f"\n【计分说明】")
    print(f"  evaluate.py 分母 = converted/ 下全部 JSON 数（当前 {total_files} 条）")
    print(f"  有标注可对比：{labeled_count} 条；Agent 命中 {correct_agent} 条；Step 命中 {correct_step} 条")
    if labeled_count and labeled_count < total_files:
        print(f"  若仅按有标注文件计分：Agent {correct_agent}/{labeled_count}，Step {correct_step}/{labeled_count}")


def run_failure_attribution_eval(
    jsonl_paths: list[Path | str],
    method: str = DEFAULT_METHOD,
    model: str = DEFAULT_MODEL,
    labels: dict[str, dict[str, str]] | None = None,
    run_inference: bool = False,
    run_evaluation: bool = False,
    eval_file: Path | str | None = None,
    skip_convert: bool = False,
) -> dict[str, Any]:
    """Run the AutoFA pipeline: convert, attribute, then optionally evaluate."""
    if method not in METHOD_CHOICES:
        raise ValueError(f"method 必须是 {METHOD_CHOICES}")

    labels = labels or {}
    jsonl_paths = [Path(p) for p in jsonl_paths]
    CONVERTED_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    ## Step 1: Convert span traces to Who&When JSON (optional mistake labels).
    converted_files: list[Path] = []
    if not skip_convert:
        for jsonl_path in jsonl_paths:
            if not jsonl_path.exists():
                raise FileNotFoundError(f"轨迹文件不存在: {jsonl_path}")
            label = labels.get(jsonl_path.stem, {})
            out_path = convert_jsonl_to_whowwhen(
                jsonl_path=jsonl_path,
                output_dir=CONVERTED_DIR,
                mistake_agent=label.get("mistake_agent"),
                mistake_step=label.get("mistake_step"),
                mistake_reason=label.get("mistake_reason"),
            )
            converted_files.append(out_path)
            _print_conversion_summary(out_path)
            print()
    else:
        converted_files = sorted(CONVERTED_DIR.glob("*.json"))

    eval_path: Path | None = None
    predictions: dict[str, Any] = {}
    labeled_files = 0
    agent_accuracy, step_accuracy = 0.0, 0.0

    need_eval_output = run_inference or run_evaluation

    ## Step 2: Attribute every converted JSON (needs DASHSCOPE_API_KEY).
    if run_inference:
        if not os.getenv("DASHSCOPE_API_KEY", "").strip():
            raise ValueError("Step 2 requires DASHSCOPE_API_KEY in the repository root .env")
        eval_path = _run_inference(method, model, CONVERTED_DIR)
    elif need_eval_output:
        eval_path = Path(eval_file) if eval_file else OUTPUT_DIR / f"{method}_{model}_alg_generated.txt"
        if not eval_path.exists():
            raise FileNotFoundError(
                f"未找到归因输出: {eval_path}\n"
                "请先运行 Step 2（run_inference=True），或取消注释 Step 2 单元格。"
            )

    ## Step 3 / Step 4: Parse predictions and, when labels exist, compute accuracy.
    if run_evaluation and eval_path is not None:
        parsed = parse_attribution_results(eval_path.read_text(encoding="utf-8"))
        predictions = {
            item["file"]: {
                "predicted_agent": item["agent"],
                "predicted_step": item["step"],
            }
            for item in parsed
        }
        total_files = len(list(CONVERTED_DIR.glob("*.json")))
        labeled_files = sum(
            1 for p in CONVERTED_DIR.glob("*.json")
            if json.loads(p.read_text(encoding="utf-8")).get("mistake_agent")
        )

        if labeled_files == 0:
            print("\n提示: converted/*.json 中无 mistake_agent 标注，跳过 Step 4 准确率计算。")
            print("可在 labels 中填写标注后重新 run_failure_attribution_eval。")
        else:
            agent_accuracy, step_accuracy = evaluate_accuracy(
                predictions, str(CONVERTED_DIR), total_files
            )

    return {
        "converted_files": [str(p) for p in converted_files],
        "eval_file": str(eval_path) if eval_path else None,
        "predictions": predictions,
        "labeled_files": labeled_files,
        "agent_accuracy": agent_accuracy,
        "step_accuracy": step_accuracy,
    }


## 2. 样例轨迹与人工标注

仓库内提供两条测试轨迹（详见 README 第六节）：

| 文件 | 用户问题 | Agent 数量 | 主要问题（人工判断） |
|------|----------|-----------|---------------------|
| `spans_20260615_170343.jsonl` | 大同未来2周的天气如何 | T1（WeatherAgent） | 仅返回 14 天预报，未覆盖完整 2 周 |
| `spans_20260615_180713.jsonl` | 上海/苏州/杭州多城市旅行规划 | T1–T4 | WeatherAgent 查询了本周天气，而非用户要求的「下周」 |

`labels` 仅在 **Step 4 评估**时使用；Step 编号等于 `history` 数组下标，从 0 开始。

In [ ]:
SAMPLE_JSONL = [
    TESTDATA_DIR / "spans_20260615_170343.jsonl",
    TESTDATA_DIR / "spans_20260615_180713.jsonl",
]

# 人工标注（README Step 4 示例：仅轨迹 2 有明确标注）
LABELS = {
    "spans_20260615_180713": {
        "mistake_agent": "WeatherAgent",
        "mistake_step": "4",
        "mistake_reason": "查询了本周天气而非用户要求的「下周」",
    },
}

for p in SAMPLE_JSONL:
    print(p.name, "→", "存在" if p.exists() else "缺失")

## 3. Step 1 — 轨迹转换

命令行等价：

```powershell
cd Chapter-9
python testdata/convert_spans_to_whowhen.py `
  testdata/spans_20260615_170343.jsonl `
  testdata/spans_20260615_180713.jsonl `
  -o testdata/converted
```

**预期结果：** 输出 2 个 JSON 文件（文件名取自输入文件 stem），文件之间不会相互覆盖。

In [ ]:
step1 = run_failure_attribution_eval(
    jsonl_paths=SAMPLE_JSONL,
    labels=LABELS,
    run_inference=False,
    run_evaluation=False,  # Step 1 只做转换，不要求 outputs/*.txt
    skip_convert=False,
)
print("已转换:", step1["converted_files"])

## 4. Step 2 — 失败归因（需 API Key，约 30–60 秒）

命令行等价：

```powershell
cd Chapter-9/Automated_FA
python inference.py `
  --method step_by_step `
  --model qwen-plus `
  --is_handcrafted False `
  --directory_path ../testdata/converted
```

输出文件：`outputs/step_by_step_qwen-plus_alg_generated.txt`

三种归因策略如下：

| 方法 | 说明 |
|------|------|
| `all_at_once` | 一次性读取完整 history，速度最快 |
| `step_by_step` | 逐步扫描，遇到第一个判定为「有错」的步骤即停止（**推荐**） |
| `binary_search` | 以二分方式定位步骤，适用于 history 较长的场景，可减少 token 消耗 |

In [ ]:
# 取消注释以运行 Step 2（会调用通义千问 API，并覆盖 outputs/*.txt）
step2 = run_failure_attribution_eval(
    jsonl_paths=SAMPLE_JSONL,
    method="step_by_step",
    model="qwen-plus",
    labels=LABELS,
    run_inference=True,
    skip_convert=True,
)
print("归因输出:", step2["eval_file"])

## 5. Step 3 — 查看归因结果

README 实测预期（`step_by_step` + `qwen-plus`）：

| 轨迹文件 | 预测 Agent | 预测 Step | 说明 |
|----------|-----------|-----------|------|
| `spans_20260615_170343.json` | `Orchestrator` | 5 | 模型在第 5 步（Orchestrator）判定有错后停止 |
| `spans_20260615_180713.json` | `WeatherAgent` | 4 | 与人工分析一致：天气日期范围错误 |

> **注意（FAQ Q2）：** `step_by_step` 在遇到**第一个**被 LLM 判定为「有错」的步骤后即停止。轨迹 1 的第 0–4 步均判定为无错，第 5 步（Orchestrator）判定为有错后停止；轨迹 2 在第 4 步（WeatherAgent）停止。对于较复杂的轨迹，建议改用 `binary_search` 或 `all_at_once`。


In [ ]:
EVAL_FILE = OUTPUT_DIR / "step_by_step_qwen-plus_alg_generated.txt"

attribution_results = print_attribution_summary_zh(EVAL_FILE, CONVERTED_DIR)

## 6. Step 4（可选）— 评估准确率

命令行等价：

```powershell
cd Chapter-9/Automated_FA
python evaluate.py `
  --data_path ../testdata/converted `
  --eval_file outputs/step_by_step_qwen-plus_alg_generated.txt
```

准确率的分母为 `converted/` 目录下全部 JSON 文件数量；在 2 条轨迹中仅有 1 条标注时，准确率上限为 50%。

In [ ]:
try:
    result = run_failure_attribution_eval(
        jsonl_paths=SAMPLE_JSONL,
        method="step_by_step",
        model="qwen-plus",
        labels=LABELS,
        run_inference=False,
        run_evaluation=True,
        skip_convert=True,
    )
except FileNotFoundError as exc:
    print(exc)
    result = {"eval_file": None, "labeled_files": 0, "predictions": {}, "agent_accuracy": 0.0, "step_accuracy": 0.0}

print("\n=== Step 4 评估结果 ===")
print(f"归因输出: {result['eval_file']}")
print(f"有标注的文件数: {result['labeled_files']}")
if result["labeled_files"]:
    print(f"Agent 准确率: {result['agent_accuracy']:.2f}%")
    print(f"Step 准确率:  {result['step_accuracy']:.2f}%")
if result["predictions"]:
    total_files = len(list(CONVERTED_DIR.glob("*.json")))
    print_evaluation_comparison_zh(result["predictions"], CONVERTED_DIR, total_files)
else:
    print("\n（无有效预测，请先完成 Step 2 生成 outputs/*.txt）")

## 7. 支持的模型与相关文件

| 类型 | `--model` 取值 | 是否需要 Key |
|------|----------------|-------------|
| 通义千问 API | `qwen-plus`, `qwen-max`, `qwen-turbo`, `qwen-long` | 需要 `DASHSCOPE_API_KEY` |
| Azure GPT | `gpt-4o`, `gpt4`, `gpt4o-mini` | 需要 `AZURE_OPENAI_*` |
| 本地 HuggingFace | `qwen-7b`, `qwen-72b`, `llama-8b`, `llama-70b` | 无需 Key，但需要 GPU |

| 文件 | 作用 |
|------|------|
| `testdata/spans_*.jsonl` | 样例轨迹 |
| `testdata/convert_spans_to_whowhen.py` | span JSONL → Who&When JSON |
| `testdata/converted/*.json` | 转换输出（归因输入） |
| `Automated_FA/inference.py` | 归因入口 |
| `Automated_FA/evaluate.py` | 准确率评估 |
| `Automated_FA/Lib/utils.py` | 三种 AutoFA 方法实现 |
| `Automated_FA/outputs/step_by_step_qwen-plus_alg_generated.txt` | 归因输出（实测生成） |
| `testdata/README.md` | 完整使用指南 |